# NB09 — Feature Ablations (Appendix D.1, D.2)

Two complementary ablations on the trained RF, both holding
hyperparameters fixed at the pkl's Optuna-tuned values
(`max_depth=19, min_samples_leaf=20, max_features=0.8`) and varying
only which columns are passed in:

**D.1 (block ablation)**: drop one of {Tech, Sector, LLM} at a time,
plus the two reverse cases (LLM-only, Tech-only). Quantifies the
val-AUC contribution of each feature block.

**D.2 (drop-one LLM)**: drop one of the six LLM features at a time
from the full 22-column matrix. Identifies which LLM feature is
actually doing the work.

These complement NB06 (permutation importance, OOS) and the
arm A/B/C backtest in NB05 (block-level on Sharpe). NB05's arm
ablation is at the Sharpe level; this notebook is at the val-AUC
level — a tighter, smaller-noise metric that gives clearer per-
feature attribution.

**Sections**
1. Load model + base data
2. D.1 — block ablation
3. D.2 — drop-one LLM feature ablation
4. Outputs

## 1. Load model + base data

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
import config
from src.llm_agent.data_loader import load_sessions, load_prices, load_llm_signals
from src.llm_agent.features    import build_feature_matrix
from src.llm_agent.ablation    import run_block_ablation, run_drop_one_llm

bundle = joblib.load(config.MODELS_DIR / 'event_study_best.pkl')
feats = bundle['features']
rf_params = bundle['best_params']

print(f'Using pkl best_params: {rf_params}')
print(f'Feature list ({len(feats)} cols), order pinned to pkl')

Using pkl best_params: {'max_depth': 19, 'min_samples_leaf': 20, 'max_features': 0.8}
Feature list (22 cols), order pinned to pkl


In [2]:
sessions = load_sessions()
prices   = load_prices()
sigs     = load_llm_signals()
base = build_feature_matrix(sessions, prices, sigs, arm='C')

car = pd.read_parquet(config.CLEAN_DIR / 'car_labels.parquet')
base = base.merge(car[['session_id','car_2_11','sigma_e']],
                  on='session_id', how='left')
base['y'] = (base['car_2_11'] > 0).astype('Int64')

print(f'feature matrix: {base.shape}')

feature matrix: (1496, 30)


## 2. D.1 — block ablation

Drop each of the three blocks (Tech / Sector / LLM) one at a time,
and report the symmetric two cases (LLM-only, Tech-only). The
"full" row is the anchor — its val AUC should match the pkl's
val_auc_tuned ≈ 0.601.

In [3]:
d1 = run_block_ablation(
    base, full_feats=feats, rf_params=rf_params,
    train_years=config.TRAIN_YEARS, val_years=config.VAL_YEARS,
)
d1.round(4)

,config,n_features,val_auc,delta_vs_full,val_top_decile_precision
0,full (Tech + Sector + LLM),22,0.6012,0.0000,0.5
1,drop Tech,17,0.5633,-0.0379,0.7
2,drop Sector,11,0.6036,0.0024,0.5
3,drop LLM,16,0.5713,-0.0298,0.7
4,LLM only,6,0.5430,-0.0581,0.7
5,Tech only,5,0.5957,-0.0055,0.6


**Interpretation guide.** A negative `delta_vs_full` means
removing that block hurts val AUC (block was contributing).
A near-zero or positive `delta_vs_full` means the block is
redundant or noisy. NB05's arm B/C comparison reports a
ΔSharpe = +0.13 / +1.07 attributable to the LLM block at the
Sharpe level; this row gives the corresponding ΔAUC at the
ranking level.

## 3. D.2 — drop-one LLM feature ablation

NB06 found that within the LLM block, only `evasion` carries
non-trivial OOS permutation importance. This ablation tests the
same hypothesis at the val-AUC level: if `evasion` is the only
LLM feature pulling weight, dropping it should cause the largest
val AUC drop, while dropping any of the other five should leave
val AUC unchanged or even improve it.

In [4]:
d2 = run_drop_one_llm(
    base, full_feats=feats, rf_params=rf_params,
    train_years=config.TRAIN_YEARS, val_years=config.VAL_YEARS,
)
d2.round(4)

,config,n_features,val_auc,delta_vs_full,val_top_decile_precision
0,full (all 6 LLM features),22,0.6012,0.0000,0.5
1,drop sentiment_num,21,0.6022,0.0010,0.5
2,drop confidence,21,0.6070,0.0058,0.7
3,drop certainty,21,0.6118,0.0106,0.6
4,drop evasion,21,0.5799,-0.0213,0.6
5,drop guidance_num,21,0.6025,0.0014,0.5
6,drop tone_num,21,0.6039,0.0027,0.5


**Reading the result.** The ranking of `delta_vs_full` magnitudes
should mirror NB06's per-feature permutation importance — both
analyses ask "what does this feature contribute" but via different
mechanics (NB06 shuffles a column at predict time; D.2 retrains
without the column). When the two ablations agree on which features
matter, the conclusion is robust to the choice of attribution method.

## 4. Outputs

In [5]:
out_dir = config.PROJECT_ROOT / 'output' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

d1.to_csv(out_dir / 'block_ablation.csv', index=False)
d2.to_csv(out_dir / 'drop_one_llm_ablation.csv', index=False)

print('Saved:')
for f in ['block_ablation.csv', 'drop_one_llm_ablation.csv']:
    print(f'  output/tables/{f}')

Saved:
  output/tables/block_ablation.csv
  output/tables/drop_one_llm_ablation.csv
